# Hemodynamic Response Functions - Interactive Tutorial

This notebook demonstrates how to use hemodynamic response functions (HRFs) in pyfmrihrf.

A hemodynamic response function (HRF) models the temporal evolution of the fMRI BOLD signal in response to a brief neural event.

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pyfmrihrf import get_hrf, gen_hrf, block_hrf, list_available_hrfs
from pyfmrihrf.hrf.functions import hrf_gaussian

# Set up plotting style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Introduction to HRFs

Let's start by exploring the available HRFs in pyfmrihrf:

In [ ]:
# List all available HRFs
available_hrfs = list_available_hrfs()
print("Available HRFs in pyfmrihrf:")
for i, hrf_name in enumerate(available_hrfs):
    print(f"{i+1:2d}. {hrf_name}")

## 2. Pre-defined HRF Objects

pyfmrihrf includes several pre-defined HRF objects. Let's examine two common examples:

In [ ]:
# Get SPM canonical HRF and Gaussian HRF
hrf_spm = get_hrf("spmg1")
hrf_gauss = get_hrf("gaussian")

print(f"SPM Canonical HRF:")
print(f"  Name: {hrf_spm.name}")
print(f"  Span: {hrf_spm.span} seconds")
print(f"  Number of basis functions: {hrf_spm.nbasis}")

print(f"\nGaussian HRF:")
print(f"  Name: {hrf_gauss.name}")
print(f"  Span: {hrf_gauss.span} seconds")
print(f"  Number of basis functions: {hrf_gauss.nbasis}")

## 3. Evaluate and Plot Basic HRFs

HRF objects are callable - you can evaluate them at specific time points:

In [ ]:
# Define time points
time_points = np.linspace(0, 25, 250)

# Evaluate the HRFs
y_spm = hrf_spm(time_points)
y_gauss = hrf_gauss(time_points)

# Scale to peak at 1.0 for comparison
y_spm_scaled = y_spm / np.max(y_spm)
y_gauss_scaled = y_gauss / np.max(y_gauss)

# Plot comparison
plt.figure(figsize=(10, 6))
plt.plot(time_points, y_spm_scaled, label='SPM Canonical', linewidth=2)
plt.plot(time_points, y_gauss_scaled, label='Gaussian', linewidth=2)
plt.xlabel('Time (seconds)')
plt.ylabel('BOLD Response (normalized)')
plt.title('Comparison of SPM Canonical and Gaussian HRFs')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Print peak times
print(f"SPM HRF peaks at: {time_points[np.argmax(y_spm)]:.2f} seconds")
print(f"Gaussian HRF peaks at: {time_points[np.argmax(y_gauss)]:.2f} seconds")

## 4. Modifying HRF Parameters

The `gen_hrf` function allows you to create HRFs with custom parameters:

In [ ]:
# Create Gaussian HRFs with different parameters
hrf_gauss_early = gen_hrf(hrf_gaussian, mean=4, sd=1, name="Early peak (4s)")
hrf_gauss_normal = gen_hrf(hrf_gaussian, mean=5, sd=2, name="Normal peak (5s)")
hrf_gauss_late = gen_hrf(hrf_gaussian, mean=7, sd=3, name="Late peak (7s)")

# Evaluate
vals_early = hrf_gauss_early(time_points)
vals_normal = hrf_gauss_normal(time_points)
vals_late = hrf_gauss_late(time_points)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(time_points, vals_early, label='Mean=4, SD=1', linewidth=2)
plt.plot(time_points, vals_normal, label='Mean=5, SD=2', linewidth=2)
plt.plot(time_points, vals_late, label='Mean=7, SD=3', linewidth=2)
plt.xlabel('Time (seconds)')
plt.ylabel('BOLD Response')
plt.title('Gaussian HRFs with Different Parameters')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Modeling Event Duration with block_hrf

Real fMRI events often have duration. The `block_hrf` function models the response to sustained events:

In [ ]:
# Create blocked HRFs with different durations
hrf_spm_instant = hrf_spm  # Original, no duration
hrf_spm_w1 = block_hrf(hrf_spm, width=1)
hrf_spm_w2 = block_hrf(hrf_spm, width=2)
hrf_spm_w4 = block_hrf(hrf_spm, width=4)

# Evaluate
resp_instant = hrf_spm_instant(time_points)
resp_w1 = hrf_spm_w1(time_points)
resp_w2 = hrf_spm_w2(time_points)
resp_w4 = hrf_spm_w4(time_points)

# Plot
plt.figure(figsize=(12, 8))

# Raw responses
plt.subplot(2, 1, 1)
plt.plot(time_points, resp_instant, label='Instantaneous', linewidth=2)
plt.plot(time_points, resp_w1, label='Width=1s', linewidth=2)
plt.plot(time_points, resp_w2, label='Width=2s', linewidth=2)
plt.plot(time_points, resp_w4, label='Width=4s', linewidth=2)
plt.xlabel('Time (seconds)')
plt.ylabel('BOLD Response')
plt.title('Effect of Event Duration on HRF Shape')
plt.legend()
plt.grid(True, alpha=0.3)

# Normalized to show shape changes
plt.subplot(2, 1, 2)
plt.plot(time_points, resp_instant/np.max(resp_instant), label='Instantaneous', linewidth=2)
plt.plot(time_points, resp_w1/np.max(resp_w1), label='Width=1s', linewidth=2)
plt.plot(time_points, resp_w2/np.max(resp_w2), label='Width=2s', linewidth=2)
plt.plot(time_points, resp_w4/np.max(resp_w4), label='Width=4s', linewidth=2)
plt.xlabel('Time (seconds)')
plt.ylabel('BOLD Response (normalized)')
plt.title('Shape Changes with Event Duration')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Normalization and Summation Control

You can control whether blocked HRFs normalize amplitude and whether they summate:

In [ ]:
# Compare normalization and summation effects
durations = [2, 4, 8]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Summation ON, Normalization OFF
ax = axes[0, 0]
for dur in durations:
    hrf = block_hrf(hrf_spm, width=dur, summate=True, normalize=False)
    ax.plot(time_points, hrf(time_points), label=f'{dur}s', linewidth=2)
ax.set_title('Summate=True, Normalize=False\n(Default behavior)')
ax.set_ylabel('BOLD Response')
ax.legend()
ax.grid(True, alpha=0.3)

# Summation ON, Normalization ON
ax = axes[0, 1]
for dur in durations:
    hrf = block_hrf(hrf_spm, width=dur, summate=True, normalize=True)
    ax.plot(time_points, hrf(time_points), label=f'{dur}s', linewidth=2)
ax.set_title('Summate=True, Normalize=True\n(Equal peak amplitudes)')
ax.legend()
ax.grid(True, alpha=0.3)

# Summation OFF, Normalization OFF
ax = axes[1, 0]
for dur in durations:
    hrf = block_hrf(hrf_spm, width=dur, summate=False, normalize=False)
    ax.plot(time_points, hrf(time_points), label=f'{dur}s', linewidth=2)
ax.set_title('Summate=False, Normalize=False\n(Saturation effect)')
ax.set_xlabel('Time (seconds)')
ax.set_ylabel('BOLD Response')
ax.legend()
ax.grid(True, alpha=0.3)

# Summation OFF, Normalization ON
ax = axes[1, 1]
for dur in durations:
    hrf = block_hrf(hrf_spm, width=dur, summate=False, normalize=True)
    ax.plot(time_points, hrf(time_points), label=f'{dur}s', linewidth=2)
ax.set_title('Summate=False, Normalize=True\n(Saturation with equal peaks)')
ax.set_xlabel('Time (seconds)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Interactive HRF Explorer

Let's create an interactive tool to explore HRF parameters:

In [ ]:
from ipywidgets import interact, FloatSlider, Dropdown

def plot_hrf_interactive(hrf_type='spmg1', duration=0, normalize=False):
    """Interactive HRF plotter."""
    # Get base HRF
    base_hrf = get_hrf(hrf_type)
    
    # Apply duration if specified
    if duration > 0:
        hrf = block_hrf(base_hrf, width=duration, normalize=normalize)
        title = f"{hrf_type.upper()} HRF with {duration}s duration"
    else:
        hrf = base_hrf
        title = f"{hrf_type.upper()} HRF (instantaneous)"
    
    # Evaluate
    t = np.linspace(0, 30, 300)
    y = hrf(t)
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.plot(t, y, 'b-', linewidth=2)
    plt.xlabel('Time (seconds)')
    plt.ylabel('BOLD Response')
    plt.title(title)
    plt.grid(True, alpha=0.3)
    
    # Add statistics
    peak_time = t[np.argmax(y)]
    peak_value = np.max(y)
    plt.text(0.02, 0.95, f'Peak: {peak_value:.3f} at {peak_time:.1f}s', 
             transform=plt.gca().transAxes, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.show()

# Create interactive widget
interact(plot_hrf_interactive,
         hrf_type=Dropdown(options=['spmg1', 'spmg2', 'spmg3', 'gamma', 'gaussian'],
                          value='spmg1', description='HRF Type:'),
         duration=FloatSlider(min=0, max=10, step=0.5, value=0, 
                             description='Duration (s):'),
         normalize=Dropdown(options=[False, True], value=False, 
                           description='Normalize:'));

## Summary

In this tutorial, you've learned how to:

1. **Use pre-defined HRFs** - Access built-in HRF models like SPM canonical and Gaussian
2. **Modify HRF parameters** - Create custom HRFs with different peak times and widths
3. **Model event durations** - Use `block_hrf` to account for sustained stimuli
4. **Control normalization and summation** - Adjust how extended events affect the HRF

These are the fundamental building blocks for fMRI analysis. In the next tutorials, we'll see how to use these HRFs to build regressors for GLM analysis.

## Exercises

Try these exercises to test your understanding:

1. Create a double gamma HRF with custom parameters
2. Compare the temporal derivatives in SPMG2 vs SPMG3
3. Explore how different precision values affect block_hrf
4. Create an HRF that peaks at exactly 6 seconds

In [ ]:
# Your exercise code here
